# Inventory Diff and Reference Generation

Runs inventory change detection and per-object reference generation in one notebook task.

In [ ]:
import sys
import os
import s3fs
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.config_utils import check_runtime_readiness, load_pipeline_config, resolve_secrets
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline.generate_parquet import concurrent_dask_ref_generation, save_ledger_after_success

import virtualizarr as vz
import obstore
import obspec_utils
print('Imports OK')

In [ ]:
check_runtime_readiness()
kp= load_pipeline_config("../configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp, dbutils)

os.environ["AWS_ACCESS_KEY_ID"] = ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET_KEY
os.environ["AWS_REGION"] = kp["s3"].get("region_name", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = kp["s3"].get("region_name", "us-east-1")
os.environ["AWS_ENDPOINT_URL"] = kp["s3"]["endpoint_url"]

## MasterLedger building

In [ ]:
ledger = build_inventory_snapshot_and_diff(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
)

print('Inventory summary:', ledger['summary'])
print('Sample new keys:', ledger['diff']['new'][:10])
print('Sample changed keys:', ledger['diff']['changed'][:10])
print('Sample deleted keys:', ledger['diff']['deleted'][:10])

inventory_diff = ledger['diff']
inventory_objects = ledger['current_objects']
pending_ledger = ledger['next_ledger']

## Reference Generation Run Mode

**IMPORTANT:** Use following cell to choose run mode:
- Dry run: generate references for one object only (verification).
- Full run: process all new/changed objects and capture runtime metrics.

In [ ]:
from datetime import datetime, UTC

# Toggle run mode here before execution.
DRY_RUN = True
FULL_RUN = not DRY_RUN

run_meta = {
    "mode": "dry-run" if DRY_RUN else "full-run",
    "selected_key": None,
    "start_time": None,
    "end_time": None,
    "duration": "00:00:00",
}
"""Dry run on new/changed dataset. Otherwise, dry run on the first dataset by alphabetical order"""
if DRY_RUN:
    candidate_keys = sorted(set(inventory_diff.get("new", []) + inventory_diff.get("changed", [])))
    if not candidate_keys:
        candidate_keys = sorted(inventory_objects.keys())

    if not candidate_keys:
        reference_generation = {
            "summary": {
                "scanned": len(inventory_objects),
                "changed_or_new": 0,
                "generated": 0,
                "skipped": 0,
                "failed": 0,
            },
            "results": [],
            "failures": [],
        }
        print("Dry-run: no new/changed objects to test.")
    else:
        test_key = candidate_keys[0]
        run_meta["selected_key"] = test_key
        print(f"Dry-run testing single key: {test_key}")

        dry_inventory_diff = {"new": [test_key], "changed": [], "deleted": [], "unchanged": []}
        dry_current_objects = {test_key: inventory_objects[test_key]}

        dry_start_time = datetime.now(UTC)
        reference_generation = concurrent_dask_ref_generation(
            kp=kp,
            access_key=ACCESS_KEY,
            secret_key=SECRET_KEY,
            inventory_diff=dry_inventory_diff,
            current_objects=dry_current_objects,
        )
        dry_end_time = datetime.now(UTC)

        run_meta["start_time"] = dry_start_time
        run_meta["end_time"] = dry_end_time
        run_meta["duration"] = str(dry_end_time - dry_start_time).split(".")[0]

else:
    """Full concurrent run on all data. Will check against MasterLedger inventory to update against diff objects only."""
    full_start_time = datetime.now(UTC)
    reference_generation = concurrent_dask_ref_generation(
        kp=kp,
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY,
        inventory_diff=inventory_diff,
        current_objects=inventory_objects,
    )
    full_end_time = datetime.now(UTC)

    run_meta["start_time"] = full_start_time
    run_meta["end_time"] = full_end_time
    run_meta["duration"] = str(full_end_time - full_start_time).split(".")[0]

print("Run mode:", run_meta["mode"])
print("Reference generation summary:", reference_generation["summary"])
print("Generated sample:", [r["key"] for r in reference_generation["results"] if r["status"] == "generated"][:10])
print("Failures:", reference_generation["failures"][:5])
print("Duration:", run_meta["duration"])

## Commit Ledger and Log Runtime

- Full run: commits ledger when no failures and appends runtime metrics to Delta.
- Dry run: skips ledger commit and runtime logging.

In [ ]:
from uuid import uuid4

runtime_log_path = "/Volumes/workspace/weather/kerchunk/runtime_log/"

if FULL_RUN:
    if reference_generation["summary"]["failed"] == 0:
        save_ledger_after_success(
            ledger_path=kp["output"]["ledger_path"],
            next_ledger=pending_ledger,
            generation_summary=reference_generation["summary"],
        )
        commit_status = "committed"
        print("Ledger committed:", kp["output"]["ledger_path"])
    else:
        commit_status = "blocked_due_to_failures"
        print("Ledger NOT committed due to generation failures.")

    # Capture end-of-pipeline time after commit decision.
    pipeline_ended_time = datetime.now(UTC)
    pipeline_start_time = run_meta["start_time"]
    total_duration = str(pipeline_ended_time - pipeline_start_time).split(".")[0]
        
# Runtime logging
    runtime_log_row = {
        "run_id": str(uuid4()),
        "mode": run_meta["mode"],
        "start_time": pipeline_start_time,
        "ended_at": pipeline_ended_time,
        "duration": total_duration,
        "scanned": int(reference_generation["summary"].get("scanned", 0)),
        "changed_or_new": int(reference_generation["summary"].get("changed_or_new", 0)),
        "generated": int(reference_generation["summary"].get("generated", 0)),
        "failed": int(reference_generation["summary"].get("failed", 0)),
        "commit_status": commit_status,
    }
# Write to DBFS as spark table
    dbutils.fs.mkdirs(runtime_log_path)
    spark.createDataFrame([runtime_log_row]).write.format("delta").mode("append").save(runtime_log_path)
    print("Runtime log appended to Delta path:", runtime_log_path)
    print("Full-run duration:", runtime_log_row["duration"])
else:
    print("Dry-run mode: ledger commit skipped and runtime log not written.")
    if reference_generation["summary"].get("failed", 0) == 0:
        print("Dry-run verification passed.")
    else:
        print("Dry-run verification failed. Inspect failures above.")

In [ ]:
fs = s3fs.S3FileSystem(key=ACCESS_KEY, secret=SECRET_KEY)
with fs.open(f"{kp['s3']['bucket']}/FINAL_DPIRD/DPIRD_final_stations.nc", "rb") as f:
    with h5py.File(f, "r") as hf:
        for name, ds in hf.items():
            print(name, ds.dtype, type(ds.fillvalue))

In [ ]:
from pipeline.generate_parquet import _build_registry
from virtualizarr.parsers import HDFParser
import traceback

key = "FINAL_DPIRD/DPIRD_final_stations.nc"
source_url = f"s3://{kp['s3']['bucket']}/{key}"
registry = _build_registry(kp, ACCESS_KEY, SECRET_KEY)
string_vars= ['station', 'code']
parser= HDFParser()    
label = parser.__class__.__name__
out = f"/Volumes/workspace/weather/kerchunk/acacia_refs/_tmp/debug_01.1.parquet"

try:
    with vz.open_virtual_dataset(
        url=source_url,
        registry=registry,
        parser=parser,
        loadable_variables=string_vars,
    ) as vds:
        vds.vz.to_kerchunk(
            filepath=out,
            format="parquet",
            record_size=100000,
            categorical_threshold=10,
        )
    print("PASS:", label, "->", out)
except Exception as exc:
    print("FAIL:", label, "->", type(exc).__name__, str(exc))
    traceback.print_exc()